# Người điều khiển thủ công: Cửa ngõ trước hành động, Phân tầng rủi ro và Ghi nhật ký kiểm toán

README cho bài học này giới thiệu Người điều khiển thủ công với một đoạn mã ngắn hỏi người dùng `APPROVE` hoặc `REJECT` sau khi tác nhân đã tạo ra một phản hồi. Mẫu này là một điểm khởi đầu tốt, nhưng các triển khai HITL thực tế thường cần ba phần bổ sung:

1. Một **cửa ngõ trước hành động** chạy **trước khi** tác nhân thực hiện một bước rủi ro, để chi phí, tính không thể đảo ngược và độ trễ được kiểm soát.
2. **Phân tầng rủi ro**, để các hành động rủi ro thấp tự động thực thi, hành động rủi ro trung bình được phê duyệt theo lô, và chỉ các hành động rủi ro cao mới chặn lại để có người điều khiển thủ công.
3. Một **nhật ký kiểm toán và vòng lặp chỉnh sửa**, để mọi quyết định cửa ngõ được ghi lại dưới dạng JSONL, và việc từ chối sẽ nhắc lại tác nhân với một lý do có cấu trúc thay vì chỉ in `Revising...`.

Notebook này xây dựng từng phần trên nền tảng các nguyên thủy giống như `06-system-message-framework.ipynb`. Nó chạy từ đầu đến cuối trong `DEMO_MODE = True` (không cần nhập tương tác) hoặc với các lời nhắc `input()` thực khi `DEMO_MODE = False`. Lưu ý: trong DEMO_MODE việc thử lại của mục tiêu thứ ba được lập trình sẵn nên cơ chế vòng lặp hiển thị rõ ràng từ đầu đến cuối. Việc phân loại lại dựa trên chỉnh sửa thực tế đòi hỏi `DEMO_MODE = False` và một nhà điều hành.

**Ngoài phạm vi (được xử lý trong các bài học khác):** xác thực và kiểm soát truy cập (bài học 06 README mối đe dọa 2), phần mềm trung gian gọi công cụ (bài học 14 phân tích sâu MAF), các mẫu tranh luận đa tác nhân.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Mẫu 1: Cổng trước hành động

Đoạn mã HITL trong README gọi agent trước, sau đó yêu cầu người dùng phê duyệt kết quả. Đó là một luồng **hậu hành động**. Agent đã thực thi, vì vậy chi phí gọi LLM đã được trả, và bất kỳ tác động phụ nào (gửi email, ghi hàng cơ sở dữ liệu, đăng bình luận) đã xảy ra.

Một luồng **trước hành động** chèn cổng trước khi agent chạy bước rủi ro. Agent đề xuất hành động, cổng quyết định có thực thi hay không, và chỉ khi được phê duyệt thì tác động phụ mới xảy ra.

| Khía cạnh | Phê duyệt hậu hành động (đoạn mã README) | Cổng trước hành động (sổ ghi chép này) |
|---|---|---|
| Khi nào phê duyệt chạy? | Sau khi agent đã tạo kết quả | Trước khi bất kỳ tác động phụ nào thực hiện |
| Chi phí LLM khi bị từ chối | Đã trả rồi | Chỉ trả cho đề xuất, không phải hành động |
| Tác động phụ không thể đảo ngược | Có thể xảy ra (hành động đã xảy ra) | Ngăn chặn được |
| Độ rõ ràng kiểm toán | Phê duyệt là câu lệnh in | Phê duyệt là bản ghi JSON kèm dấu thời gian, hành động, lý do |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Mẫu 2: Phân tầng rủi ro

Không phải mọi hành động đều cần sự phê duyệt của con người. Việc tra cứu chỉ đọc qua một API công khai có mức độ quan trọng khác so với việc gửi email cho khách hàng. Xử lý cả hai như nhau sẽ lãng phí sự chú ý của người điều hành và làm chậm tác nhân.

Một mô hình đơn giản gồm 3 tầng:

| Tầng | Ví dụ | Quy trình phê duyệt |
|---|---|---|
| `low` (chỉ đọc) | Tìm kiếm cơ sở tri thức, tra cứu các lựa chọn chuyến bay, lấy trang web công khai | Tự động thực thi, ghi lại để kiểm tra |
| `medium` (thay đổi nhẹ) | Lưu trữ kết quả vào bộ nhớ đệm, tăng bộ đếm, lên lịch nhắc nhở | Tự động thực thi, nhưng xem xét theo lô hàng ngày |
| `high` (liên quan đến bên ngoài hoặc không thể đảo ngược) | Gửi email, tính phí thẻ, đăng lên kênh công khai | Chặn chờ phê duyệt của con người |

Đây là một cách phân tầng. Hệ thống sản xuất thường sử dụng các tầng chi tiết hơn (ví dụ, các cấp quyền IAM của AWS, các tầng truy cập dựa trên vai trò). Phiên bản 3 tầng dưới đây là phiên bản nhỏ nhất có ích cho một tác nhân kết hợp các hành động chỉ đọc và có tác động phụ.

Bộ phân loại dưới đây sử dụng các quy tắc từ khóa nhằm giữ cho bản demo mang tính xác định và chi phí thấp. Trong một hệ thống sản xuất, bạn sẽ thay bằng bộ phân loại học được hoặc một engine chính sách.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Mẫu 3: Nhật ký kiểm toán và vòng lặp sửa đổi

Một `print("Response approved.")` không phải là nhật ký kiểm toán. Để đảm bảo sự tin cậy, mỗi quyết định của cổng nên được ghi lại dưới dạng một sự kiện có cấu trúc mà bạn có thể truy vấn, phát lại hoặc đính kèm vào một cuộc đánh giá sự cố sau này.

Hai phần:

1. **JSONL chỉ thêm.** Mỗi dòng là một quyết định, kèm theo dấu thời gian, hành động, cấp độ, quyết định, lý do. Dễ dàng tìm kiếm, dễ dàng gửi đến một kho lưu trữ nhật ký thực tế sau này.
2. **Vòng lặp sửa đổi khi bị từ chối.** Khi cổng trả về `deny`, đại lý sẽ tự yêu cầu lại với lý do từ chối trong ngữ cảnh, để đề xuất tiếp theo có thể tránh được sự cố.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Tài nguyên bổ sung

Một số dự án công khai khác triển khai các biến thể của các mẫu HITL này. So sánh các phương pháp để tìm ra cách phù hợp với ngăn xếp của bạn:

- **LangChain** các wrapper công cụ có người tham gia xử lý ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): các wrapper công cụ có thể cắm vào ngắt thực thi để lấy đầu vào của con người.
- **AutoGen** `UserProxyAgent` ([tài liệu v0.2](https://microsoft.github.io/autogen/0.2/docs/tutorial/human-in-the-loop); AutoGen v0.4+ đã tái cấu trúc điều này): sử dụng một vai trò agent đại diện cho con người trong các cuộc hội thoại đa tác nhân.
- **Microsoft Agent Framework (MAF)** middleware gọi hàm ([docs](https://learn.microsoft.com/agent-framework/)): middleware chạy quanh mỗi lần gọi công cụ/hàm, phù hợp để kiểm soát luồng logic và phê duyệt.

Mỗi dự án xử lý ba mẫu con này khác nhau: LangChain đóng gói chúng dưới dạng công cụ, AutoGen sử dụng vai trò agent, còn Microsoft Agent Framework sử dụng middleware gọi hàm. Đọc một hoặc hai triển khai toàn bộ trước khi chọn thiết kế cho agent của bạn.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
